# Smart MCQ Solver — Baseline 1: TF-IDF + Deep Neural Network (from scratch, PyTorch)

**Pipeline:** `prompt + options -> TF-IDF sparse vector -> 3-layer MLP -> softmax over {A,B,C,D,E} -> top-3 for MAP@3`

Tracked with Weights & Biases. Runs on Kaggle GPU (T4/P100) via `torch.cuda.amp` mixed precision.


### Fix: `AcceleratorError: no kernel image is available for execution on the device`
This is not a code bug -- it means the installed PyTorch build's compiled CUDA kernels don't match the
driver/GPU on the machine. On Kaggle the #1 cause is `pip install <package>` silently resolving a *different*
torch wheel than the one preloaded and matched to the image's driver. Fix: install extra packages with
`--no-deps` so pip never touches torch/numpy, and verify the GPU with a real tensor op (not just
`torch.cuda.is_available()`, which returns `True` even when the installed kernels can't actually run on the
card) before training -- falling back to CPU automatically if it fails.


In [9]:
# --no-deps is the important part: stops pip from upgrading/downgrading torch to satisfy wandb's requirements
!pip install -q --no-deps wandb

import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import wandb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("torch:", torch.__version__, "| built for CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def get_device():
    """torch.cuda.is_available() can be True even when the installed kernels can't run on this
    GPU (exactly the error above) -- the only real test is running an actual op."""
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        (torch.zeros(1, device="cuda") + 1).cpu()   # tiny real op, not just an availability check
        return torch.device("cuda")
    except Exception as e:
        print("CUDA reports available but a test op failed, falling back to CPU.")
        print("Underlying error:", e)
        print("Fix on Kaggle: Settings -> Accelerator -> reselect the GPU, or Factory reset the session,")
        print("since this happens when a pip install swapped in a torch build the driver doesn't match.")
        return torch.device("cpu")

device = get_device()
print("Using device:", device)


torch: 2.10.0+cu128 | built for CUDA: 12.8
GPU: Tesla P100-PCIE-16GB
CUDA reports available but a test op failed, falling back to CPU.
Underlying error: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Fix on Kaggle: Settings -> Accelerator -> reselect the GPU, or Factory reset the session,
since this happens when a pip install swapped in a torch build the driver doesn't match.
Using device: cpu


In [10]:
CONFIG = {
    "max_features": 20000,
    "ngram_range": (1, 2),
    "hidden1": 512,
    "hidden2": 256,
    "dropout": 0.3,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 128,
    "epochs": 15,
    "val_split": 0.15,
}

wandb.login()  # or set WANDB_API_KEY as a Kaggle secret and it will pick it up silently
wandb.init(project="smart-mcq-solver", name="tfidf-dnn-baseline", config=CONFIG)


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


### Load data and build one text field per question
We concatenate the prompt with all 5 options, tagged, into a single string. This is a single **5-way multiclass**
problem (predict which letter is correct) rather than 5 separate binary problems — simpler, and MAP@3 falls
out directly from ranking the 5 softmax probabilities.


In [11]:
train_df = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/train.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/test.csv")

LABELS = ["A", "B", "C", "D", "E"]
label2idx = {l: i for i, l in enumerate(LABELS)}

def build_text(df):
    return (
        df["prompt"].fillna("") + " [A] " + df["A"].fillna("") +
        " [B] " + df["B"].fillna("") + " [C] " + df["C"].fillna("") +
        " [D] " + df["D"].fillna("") + " [E] " + df["E"].fillna("")
    )

train_df["text"]  = build_text(train_df)
test_df["text"]   = build_text(test_df)
train_df["label"] = train_df["answer"].map(label2idx)

train_df[["text", "label"]].head(2)


,text,label
0,Pick the best possible answer: What is Martin ...,1
1,What is accelerator-based light-ion fusion? [A...,0


In [12]:
tfidf = TfidfVectorizer(
    max_features=CONFIG["max_features"],
    ngram_range=CONFIG["ngram_range"],
    sublinear_tf=True,
    stop_words="english",
)

# fit on train+test text so the vocabulary covers test-time words too (transductive, no label leakage)
tfidf.fit(pd.concat([train_df["text"], test_df["text"]]))

X_all  = tfidf.transform(train_df["text"])   # scipy sparse csr, shape (N_train, V)
y_all  = train_df["label"].values
X_test = tfidf.transform(test_df["text"])

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=CONFIG["val_split"], random_state=SEED, stratify=y_all
)

print("Vocab size:", len(tfidf.vocabulary_))
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Vocab size: 12311
Train: (1700, 12311) Val: (300, 12311) Test: (500, 12311)


In [13]:
class TfidfDataset(Dataset):
    """Densifies one sparse row at a time so we never materialize the full dense matrix."""
    def __init__(self, X_sparse, y=None):
        self.X = X_sparse
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        row = torch.from_numpy(self.X[idx].toarray().squeeze(0)).float()  # (V,)
        if self.y is not None:
            return row, torch.tensor(self.y[idx], dtype=torch.long)
        return row

train_loader = DataLoader(TfidfDataset(X_train, y_train), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader   = DataLoader(TfidfDataset(X_val, y_val),   batch_size=CONFIG["batch_size"], shuffle=False)
test_loader  = DataLoader(TfidfDataset(X_test),         batch_size=CONFIG["batch_size"], shuffle=False)


In [14]:
class TfidfDNN(nn.Module):
    def __init__(self, vocab_size, hidden1, hidden2, n_classes=5, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vocab_size, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, n_classes),   # raw logits; CrossEntropyLoss applies softmax internally
        )

    def forward(self, x):
        return self.net(x)

model = TfidfDNN(
    vocab_size=X_train.shape[1],
    hidden1=CONFIG["hidden1"],
    hidden2=CONFIG["hidden2"],
    dropout=CONFIG["dropout"],
).to(device)

wandb.watch(model, log="all", log_freq=50)
print(model)


TfidfDNN(
  (net): Sequential(
    (0): Linear(in_features=12311, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=5, bias=True)
  )
)


In [15]:
def map3_score(logits, targets):
    top3 = torch.topk(logits, k=3, dim=1).indices  # (batch, 3)
    score = 0.0
    for pred_row, true in zip(top3.tolist(), targets.tolist()):
        if true in pred_row:
            score += 1.0 / (pred_row.index(true) + 1)
    return score / len(targets)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
# torch.cuda.amp.* is deprecated in favor of the device-agnostic torch.amp.* API
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = total_acc = total_map3 = 0.0
    n_batches = 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
                logits = model(X_batch)
                loss = criterion(logits, y_batch)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = logits.argmax(dim=1)
        total_loss += loss.item()
        total_acc  += (preds == y_batch).float().mean().item()
        total_map3 += map3_score(logits.detach(), y_batch)
        n_batches  += 1

    return total_loss / n_batches, total_acc / n_batches, total_map3 / n_batches


In [16]:
best_val_map3 = 0.0
for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc, train_map3 = run_epoch(train_loader, train=True)
    val_loss,   val_acc,   val_map3   = run_epoch(val_loader,   train=False)

    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss, "train/acc": train_acc, "train/map3": train_map3,
        "val/loss": val_loss,     "val/acc": val_acc,     "val/map3": val_map3,
    })

    if val_map3 > best_val_map3:
        best_val_map3 = val_map3
        torch.save(model.state_dict(), "best_model.pt")

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} val_loss {val_loss:.4f} | val_map3 {val_map3:.4f}")

print("Best val MAP@3:", best_val_map3)


Epoch 01 | train_loss 0.5163 val_loss 1.4763 | val_map3 0.5681
Epoch 02 | train_loss 0.0292 val_loss 0.9439 | val_map3 0.9081
Epoch 03 | train_loss 0.0097 val_loss 0.2970 | val_map3 1.0000
Epoch 04 | train_loss 0.0057 val_loss 0.0423 | val_map3 1.0000
Epoch 05 | train_loss 0.0042 val_loss 0.0070 | val_map3 1.0000
Epoch 06 | train_loss 0.0032 val_loss 0.0027 | val_map3 1.0000
Epoch 07 | train_loss 0.0027 val_loss 0.0017 | val_map3 1.0000
Epoch 08 | train_loss 0.0024 val_loss 0.0014 | val_map3 1.0000
Epoch 09 | train_loss 0.0020 val_loss 0.0012 | val_map3 1.0000
Epoch 10 | train_loss 0.0019 val_loss 0.0011 | val_map3 1.0000
Epoch 11 | train_loss 0.0018 val_loss 0.0009 | val_map3 1.0000
Epoch 12 | train_loss 0.0016 val_loss 0.0009 | val_map3 1.0000
Epoch 13 | train_loss 0.0015 val_loss 0.0008 | val_map3 1.0000
Epoch 14 | train_loss 0.0013 val_loss 0.0007 | val_map3 1.0000
Epoch 15 | train_loss 0.0011 val_loss 0.0007 | val_map3 1.0000
Best val MAP@3: 1.0


In [17]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

all_top3 = []
with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        top3 = torch.topk(logits, k=3, dim=1).indices.cpu().tolist()
        all_top3.extend(top3)

pred_labels = [" ".join(LABELS[i] for i in row) for row in all_top3]

submission = pd.DataFrame({"ID": test_df["id"], "Prediction": pred_labels})
submission.to_csv("submission.csv", index=False)
submission.head()


,ID,Prediction
0,1,A D B
1,2,B D A
2,3,B D A
3,4,E D A
4,5,C D B


In [18]:
wandb.finish()


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train/acc,▁██████████████
train/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/map3,▁██████████████
val/acc,▁▆█████████████
val/loss,█▅▂▁▁▁▁▁▁▁▁▁▁▁▁
val/map3,▁▇█████████████
epoch,15
train/acc,1
train/loss,0.00109
train/map3,1
